# Genetic Algorithm Hyperparameter Tuning
This notebook will show the outputs of the hyperparameter tuning. So this will be done in stages rather than a full grid search. In addition this will be done for only the moderate city, this is because if we used all three cities during tuning would conflate city structure with hyperparameter effects. You wouldn't know if a parameter setting is better because it's genuinely better or just better suited to that specific city.

Moderate city is the right choice for tuning because it's the middle ground — grid is too simple and bottleneck is too constrained. Once you've 

In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import pickle

def extract_results(path):
    with open(path, "rb") as f:
        results = pickle.load(f)
    
    return results

In [19]:
import numpy as np
import pandas as pd


def summarise_experiment(results: dict, experiment_name: str) -> dict:
    evals = results["evals"]
    all_metrics = [e.metrics for e in evals]

    summary = {"experiment": experiment_name}
    for metric in all_metrics[0].keys():
        values = [m[metric] for m in all_metrics]
        summary[f"{metric} (mean)"] = round(float(np.mean(values)), 3)
        summary[f"{metric} (std)"] = round(float(np.std(values)), 3)

    return summary


def compare_experiments(filepaths: dict) -> pd.DataFrame:
    """
    filepaths: dict of {experiment_name: filepath}
    e.g. {
        "fitness_max": "results/fitness_max.pkl",
        "fitness_mean": "results/fitness_mean.pkl",
        "fitness_05max_05mean": "results/fitness_combo_05.pkl",
    }
    """
    rows = []
    for name, path in filepaths.items():
        results = extract_results(path)
        summary = summarise_experiment(results, name)
        rows.append(summary)

    df = pd.DataFrame(rows).set_index("experiment")
    return df


def print_comparison(df: pd.DataFrame, sort_by: str = "Total time (mean)") -> None:
    sorted_df = df.sort_values(sort_by)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    print(sorted_df.to_string())

### Fitness Function
The following parameters were tried first:
```
{"fitness": "max", "alpha": 1},
{"fitness": "mean", "alpha": 1},
{"fitness": "median", "alpha": 1},
{"fitness": "max-median", "alpha": 0.8},
{"fitness": "max-median", "alpha": 0.65},
{"fitness": "max-median", "alpha": 0.5}
```

Saved under the following name:
```
path = f"outputs/tuning/{city.city_name}_{n_experiments}_{params['fitness']}_{params['alpha']}_fitness.pkl"

>>outputs/tuning/moderate_city_3_{params['fitness']}_{params['alpha']}_fitness.pkl

```

In [22]:
filepaths = {
    "fitness_max":          "outputs/tuning/moderate_city_3_max_1_fitness.pkl",
    "fitness_mean":         "outputs/tuning/moderate_city_3_mean_1_fitness.pkl",
    "fitness_median":         "outputs/tuning/moderate_city_3_median_1_fitness.pkl",
    "fitness_08max_02median": "outputs/tuning/moderate_city_3_max-median_0.8_fitness.pkl",
    "fitness_065max_035median":       "outputs/tuning/moderate_city_3_max-median_0.65_fitness.pkl",
    "fitness_05max_05median": "outputs/tuning/moderate_city_3_max-median_0.5_fitness.pkl",
}


In [ ]:
df = compare_experiments(filepaths)
print_comparison(df, sort_by="Total time (mean)")

                          Avg path length (mean)  Avg path length (std)  Total time (mean)  Total time (std)  Average time (mean)  Average time (std)  Congestion index (mean)  Congestion index (std)  Avg Congestion delay (mean)  Avg Congestion delay (std)  Exit utilisation (mean)  Exit utilisation (std)  Avg Path Efficiency (mean)  Avg Path Efficiency (std)
experiment                                                                                                                                                                                                                                                                                                                                                             
fitness_08max_02median                     8.980                  0.073             16.667             0.471               11.367               0.209                    0.857                   0.031                        2.387                       0.144                    0.092

In [31]:
df[['Avg path length (mean)', 'Total time (mean)', 'Average time (mean)', 'Congestion index (mean)', 
    'Avg Congestion delay (mean)', 'Exit utilisation (mean)', 'Avg Path Efficiency (mean)']].sort_values('Avg Congestion delay (mean)')


,Avg path length (mean),Total time (mean),Average time (mean),Congestion index (mean),Avg Congestion delay (mean),Exit utilisation (mean),Avg Path Efficiency (mean)
experiment,,,,,,,
fitness_mean,8.187,17.667,10.493,0.833,2.307,0.100,1.105
fitness_median,8.970,25.333,11.277,0.830,2.307,0.112,1.198
fitness_05max_05median,8.997,17.000,11.360,0.840,2.363,0.075,1.180
fitness_08max_02median,8.980,16.667,11.367,0.857,2.387,0.092,1.164
fitness_065max_035median,8.833,17.000,11.260,0.827,2.427,0.105,1.150
fitness_max,9.207,17.000,11.640,0.827,2.433,0.065,1.182


Clear winner is fitness_08max_02mean:
* Lowest total time (16.667) — the only config that gets below 17
* Lowest std on total time (0.471) — most consistent across seeds
* Lowest avg congestion delay (2.387)
* Best path efficiency (1.164) — agents taking reasonably direct routes

A few other observations:
* fitness_mean has the best average time (10.493) and shortest paths (8.187) but total time is worst (17.667) — it's optimising average at the expense of stragglers, which is wrong for evacuation
* fitness_median is clearly worst — total time 25.333 is terrible, drop it entirely
* fitness_065max has the best exit utilisation (0.105) but total time is inconsistent (std 0.816)
* fitness_max is completely flat (std 0.000) — confirms the earlier finding that pure max gives no gradient

### Epsilon
The following parameters were tried for the mutation greedy path and utilising the above parameters for fitness:
```
epsilon - [0.2, 0.4, 0.6, 0.8]
```

Saved under the following name:
```
path = f"outputs/tuning/{city.city_name}_{n_experiments}_{params['epsilon']}_epsilon.pkl"
>>outputs/tuning/moderate_city_3_{params['epsilon']}_epsilon.pkl
```